# Solow Growth Model using Dynare and GNU Octave

This notebook demonstrates how to simulate the Solow growth model using Dynare within a computational environment.

The objective is to connect:
- macroeconomic theory
- numerical simulation

We focus on how the economy transitions between steady states after a structural change.

## Step 1: Setting up the Environment

We install the necessary tools:

- **GNU Octave**: numerical computing environment (similar to MATLAB)
- **Dynare**: specialized software for solving macroeconomic models

This allows us to run `.mod` files that describe dynamic economic systems.

In [ ]:
!apt-get update -qq
!apt-get install -y octave dynare

## Step 2: Testing Dynare Installation

Before running a full model, we test whether Dynare is working correctly.

### Test Equation
$$ y_t = 0.9 y_t $$

This equation has no economic interpretation.

### Purpose
- verify installation
- ensure `.mod` files execute correctly


In [ ]:
%%bash

cat <<EOF > test.mod
var y;
model;
y = 0.9*y;
end;
EOF

octave --eval "addpath('/usr/lib/dynare/matlab'); dynare test.mod"

## Step 3: Solow Growth Model

We now implement the Solow growth model, a fundamental framework in macroeconomics.

### Production Function
$$ y_t = A k_{t-1}^{\alpha} $$

- Output depends on capital
- $A$ captures productivity

---

### Capital Accumulation
$$ k_t = \frac{s y_t + (1-\delta)k_{t-1}}{(1+n)(1+g)} $$

- Savings determines investment
- Depreciation reduces capital
- Population and technology dilute capital

---

### Key Insight

Due to diminishing returns:

> The economy converges to a steady state.
> Higher savings raises output levels, but not long-run growth.

In [ ]:
%%bash

cat <<EOF > solow.mod
var y k c w r sav;
varexo a x z;

parameters alpha delta n g s;

alpha = 0.333;
delta = 0.03;
n = 0.01;
g = 0.02;
s = 0.30;

model;
y = a*(k(-1)^alpha);
c = (1-(s*x))*y;
k = (1/((1+n*z)*(1+g)))*((s*x*y) + (1-delta)*k(-1));
r = alpha*a*k(-1)^(alpha-1) - delta;
w = (1-alpha)*a*k(-1)^alpha;
sav = s*x;
end;

initval;
k = 5.3;
c = 1.38;
y = 1.7;
a = 1;
r = 0.075;
w = 1.16;
x = 1;
z = 1;
end;

steady;

endval;
z = 1.01;
end;

steady;
check;

perfect_foresight_setup(periods=100);
perfect_foresight_solver;

save('solow_results.mat', 'oo_');
EOF

octave --eval "addpath('/usr/lib/dynare/matlab'); dynare solow.mod"

## Step 4: Transition Dynamics

We introduce a structural change using:
`endval`

Dynare computes how the economy transitions over time.

### Interpretation
- The economy starts in steady state
- A shock shifts the system
- Adjustment occurs gradually over time

This reflects real-world capital accumulation processes.

In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt

data = sio.loadmat('solow_results.mat')
endo = data['oo_']['endo_simul'][0][0]

names = ['y','k','c','w','r','sav']

for i, name in enumerate(names):
    plt.figure()
    plt.plot(endo[i])
    plt.title(name)
    plt.xlabel('Time')
    plt.show()

## Step 5: Interpreting Results

The simulation highlights the main predictions of the Solow model:

- Capital increases gradually
- Output and wages rise with capital
- Consumption adjusts with savings behavior
- Interest rates decline due to diminishing returns

---

### Core Result

> Changes in saving affect income levels,
> but not long-run growth rates.

The economy converges smoothly to a new steady state.